In [ ]:
import os
hsaDir = os.path.abspath("../../")

In [ ]:
from heliostat_surface_analysis.loader.csv_loader import CSVLoader
from heliostat_surface_analysis.preprocessor.basic_preprocessor import BasicPreprocessor
from heliostat_surface_analysis.mesher.delaunay_mesher import DelaunayMesher
from heliostat_surface_analysis.mesher.bspline_mesher import BSplineMesher
from heliostat_surface_analysis.analyser.fit_surface_analyser import FitSurfaceAnalyser
from heliostat_surface_analysis.aggregator.aggregator import WeightedMLEAggregator
from heliostat_surface_analysis.analysis_manager import AnalysisManager

In [ ]:
import pandas as pd
import progressbar

In [ ]:
%matplotlib inline

In [ ]:
# for progressbar
widgets = [' [',
         progressbar.Timer(format= 'elapsed time: %(elapsed)s'),
         '] ',
           progressbar.Bar('*'),' (',
           progressbar.ETA(), ') ',
          ]

### Setup analysis components

In [ ]:
preprocessor = BasicPreprocessor()
meshers = {}
# NOTE: setting the max_aspect_ratio is important
meshers["Delaunay"] = DelaunayMesher({'max_aspect_ratio':10})
meshers["B-spline"] = BSplineMesher({'polynomial_degree':3,'dx':0.01,'dy':0.01})
analyser = FitSurfaceAnalyser({"surface":"flat"})
aggregator = WeightedMLEAggregator()

In [ ]:
caseName = "CSIRO Solar Field 1 reflector"

In [ ]:
if caseName=="CSIRO Solar Field 1 reflector":
    caseDir = os.path.join(hsaDir,"examples","flat","CSIRO_SolarField1_reflector")
    loader = CSVLoader({"units":"mm"})
else:
    raise RuntimeError("caseName: %s not recognised"%caseName)

In [ ]:
scansDir = os.path.join(caseDir,"scans")
scanFiles = sorted(os.listdir(scansDir))
print(len(scanFiles))

### Compare meshers

In [ ]:
scanNames = []
slopeErrors = {}
for mesherName,_ in meshers.items():
    slopeErrors[mesherName] = []
bar = progressbar.ProgressBar(maxval=len(scanFiles)).start()
for i,scanFile in enumerate(scanFiles):
    scanPath = os.path.join(scansDir,scanFile)
    for mesherName,mesher in meshers.items():
        result = AnalysisManager.analyse_scan(scanPath,loader,preprocessor,mesher,analyser,aggregator,output_dir=None,plot=False,write_stls=False)
        slopeErrors[mesherName].append(result["aggregate"])
        scanName = result["name"]
    # these are the same irrespective of what mesher is used
    scanNames.append(scanName)
    bar.update(i+1)
bar.finish()

In [ ]:
dataDict = {}
dataDict["scan"] = scanNames
for mesherName,mesher in meshers.items():
    dataDict[f"sigma ({mesherName})"] = slopeErrors[mesherName]
df = pd.DataFrame(dataDict)
df.set_index("scan",inplace=True)
df.head(n=150)

### Compare analysers

In [ ]:
mesher = meshers["Delaunay"]
scanNames = []
slopeErrors = []
bar = progressbar.ProgressBar(maxval=len(scanFiles)).start()
for i,scanFile in enumerate(scanFiles):
    scanPath = os.path.join(scansDir,scanFile)
    result = AnalysisManager.analyse_scan(scanPath,loader,preprocessor,mesher,analyser,aggregator,output_dir=None,plot=False,write_stls=False)
    slopeErrors.append(result["aggregate"])
    scanName = result["name"]
    scanNames.append(scanName)
    bar.update(i+1)
bar.finish()

In [ ]:
dataDict = {}
dataDict["scan"] = scanNames
dataDict["sigma"] = slopeErrors
df = pd.DataFrame(dataDict)
df.set_index("scan",inplace=True)
with pd.ExcelWriter('facets_analysis.ods', engine="odf") as doc:
    df.to_excel(doc, sheet_name="Sheet1")
df.head(n=150)

In [ ]:
print("mean slope error = %f mrad"%(df["sigma"].mean()*1e3))